In [1]:
import cvxpy as cp
import numpy as np
import mosek
from scipy.special import logsumexp

from bayesian_dro.Bayesian_DRO_continuous import cost, data_generation, theta_generation, xi_generation, main_Bayesian_DRO

from mis_dro.constants import NUM_OBSERVATIONS, NUM_LIKELIHOOD_SAMPLES, NUM_POSTERIOR_SAMPLES, NUM_TEST_OBSERVATIONS


In [2]:
def cvxpy_cost(x, xi):
    b = 8
    h = 3
    return h * cp.maximum(0, x - xi) + b * cp.maximum(0, xi - x)

def bdro(xi, epsilon) -> float:
    """Bayesian DRO as a cvxpy optimisaton problem"""
    num_theta_samples = xi.shape[0]
    num_xi_samples = xi.shape[1]

    x = cp.Variable(1, name="x")
    # lam = [cp.Variable(1, name=f"lam_{i}", nonneg=True) for i in range(num_theta_samples)]
    lam = cp.Variable(1, name="lam", nonneg=True)

    # Xi = cp.Variable(xi.shape, name="Xi")

    objective = cp.Minimize(
        (1.0/num_theta_samples) * cp.sum([
            lam * epsilon
            + lam * cp.log(1.0 / num_xi_samples)
            + cp.perspective(cp.log_sum_exp(cvxpy_cost(x, xi[i])), lam)
            for i in range(num_theta_samples)
        ])
    )
    constraints = [
        x >= 5,
        x <= 50,
        # Xi == xi,
    ]
    print(cp.log_sum_exp(cvxpy_cost(x, xi[0])).variables())
    print(cp.log_sum_exp(cvxpy_cost(x, xi[0])).constants())
    problem = cp.Problem(objective, constraints)
    obj_value = problem.solve(solver="MOSEK")
    print("Value of x:", x.value)
    print("Value of lambda:", lam.value)
    print("Objective:", obj_value)
    print(
        lam.value * epsilon + lam.value * np.log(1.0 / num_xi_samples) + lam.value * logsumexp(cost(x.value, xi[0]) / lam.value),
        lam.value * epsilon + lam.value * np.log(1.0 / num_xi_samples) + lam.value * logsumexp(cost(x.value / lam.value, xi[0]))
    )
    return x.value[0]

In [3]:
generator = np.random.default_rng(seed=2)
data = data_generation(NUM_OBSERVATIONS, random_state=generator)
data_eval = data_generation(NUM_TEST_OBSERVATIONS, random_state=generator)
theta = theta_generation(data, NUM_POSTERIOR_SAMPLES, random_state=generator)
xi = np.zeros([NUM_POSTERIOR_SAMPLES, NUM_LIKELIHOOD_SAMPLES])
for i in range(NUM_POSTERIOR_SAMPLES):
    xi[i] = xi_generation(theta[i], NUM_LIKELIHOOD_SAMPLES, random_state=generator)

theta.shape, xi.shape

((100,), (100, 100))

In [4]:
epsilon = 1.0
x_new = bdro(xi, epsilon)
cost_new = cost(x_new, data_eval)
cost_new.mean(), cost_new.var()

[Variable((1,), x)]
[Constant(CONSTANT, NONNEGATIVE, ()), Constant(CONSTANT, ZERO, ()), Constant(CONSTANT, NONNEGATIVE, (100,)), Constant(CONSTANT, NONNEGATIVE, ()), Constant(CONSTANT, ZERO, ()), Constant(CONSTANT, NONNEGATIVE, (100,))]
Value of x: [5.]
Value of lambda: [0.06548964]
Objective: 14.825694394163536
[373.80138153] [14.7363546]


(67.66215363268206, 2647.88765592702)

In [5]:
x_main = main_Bayesian_DRO(xi, epsilon)
cost_main = cost(x_main, data_eval)
cost_main.mean(), cost_main.var()

(61.835065615749166, 450.98663124602217)

In [6]:
logsumexp(cvxpy_cost(x_new, xi[0]).value)

374.03748284463194